# 🌧️ Rainfall dashboards — run locally on your shapefile

This notebook builds the rainfall dataset for **your** polygon shapefile and generates one
static **HTML dashboard per polygon** (the same content as the Streamlit app: May–Dec 10-year
rainy-days & cumulative-rainfall heatmaps for JAXA GSMaP + IMD, the current-year view, a live
WeatherNext forecast, a map, and the weekly table).

**Prerequisites** (once):
```bash
pip install -r requirements-build.txt   # geopandas, imdlib, plotly, folium, streamlit, ...
```

**How it works:** edit the config cell, then run the cells top to bottom.
- **Step 1 (build)** downloads GSMaP (JAXA FTP, parallel) + IMD (imdlib) for your polygons and
  bakes `data/out/`. This is the slow step (~15–30 min the first time); set `BUILD_DATA = False`
  to reuse an already-built `data/out/`.
- **Step 2 (generate)** writes `html_out/<id>.html` for every polygon + an `index.html`.

## Config — edit these

In [ ]:
# --- paths -------------------------------------------------------------
APP_DIR   = "/Users/mipl/Documents/Agroforestry/rainfall_app"   # this repo
SHAPEFILE = "/path/to/your/blocks.shp"                          # <-- YOUR shapefile

# --- shapefile columns -------------------------------------------------
BLOCK_COL    = "Block"        # polygon name/id column (required)
DISTRICT_COL = "District_x"   # optional grouping column ( "" if none )
STATE_COL    = "State_x"      # optional grouping column ( "" if none )
ID_COL       = "block"        # column used for each HTML filename (unique)

# --- analysis options --------------------------------------------------
SOURCES    = ["gsmap", "imd"]          # rainfall sources to include
THRESHOLD  = 1.0                        # rainy-day threshold (mm)
YEARS      = list(range(2016, 2026))   # 10-year historical window
OUT_DIR    = "html_out"                # output folder (under APP_DIR)
WORKERS    = 8                          # parallel FTP connections for GSMaP

# Step 1: build rainfall data for THIS shapefile?  (slow, one-time per shapefile)
#   True  -> download + bake data/out/ for your polygons
#   False -> reuse whatever is already in data/out/
BUILD_DATA = True

import os, sys, subprocess
os.chdir(APP_DIR)
print("working dir:", os.getcwd())

## Step 1 — build the rainfall dataset for your polygons
Downloads GSMaP + IMD and writes `data/out/`. Skipped when `BUILD_DATA = False`.
(Re-run any time to refresh the current-year data; already-downloaded days are cached.)

In [ ]:
if BUILD_DATA:
    cmd = [sys.executable, "build_dataset.py",
           "--shp", SHAPEFILE,
           "--block-col", BLOCK_COL, "--district-col", DISTRICT_COL, "--state-col", STATE_COL,
           "--sources", *SOURCES,
           "--years", *map(str, YEARS),
           "--workers", str(WORKERS)]
    print("Running:", " ".join(cmd), "\n")
    subprocess.run(cmd, check=True)
else:
    print("Skipping build — reusing existing data/out/")

## Step 2 — generate one HTML per polygon

In [ ]:
cmd = [sys.executable, "generate_html.py",
       "--sources", *SOURCES,
       "--threshold", str(THRESHOLD),
       "--id-col", ID_COL,
       "--out-dir", OUT_DIR]
print("Running:", " ".join(cmd), "\n")
subprocess.run(cmd, check=True)

## Step 3 — open the results
Lists the generated files and gives a clickable link to the index. Or serve the folder with
`python -m http.server 8000 -d html_out` and open <http://localhost:8000>.

In [ ]:
from pathlib import Path
from IPython.display import FileLink
out = Path(APP_DIR) / OUT_DIR
files = sorted(p.name for p in out.glob("*.html") if p.name != "index.html")
print(f"{len(files)} block pages in {out}")
for f in files[:15]:
    print("  ", f)
if len(files) > 15:
    print(f"  ... and {len(files) - 15} more")
FileLink(str(out / "index.html"))

In [ ]:
# Optional: preview the index inline (or open the FileLink above in a new tab)
from IPython.display import IFrame
IFrame(src=f"{OUT_DIR}/index.html", width="100%", height=500)